In [ ]:
import torch

print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

True
Tesla T4


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
%cd /content/drive/MyDrive/Colab_Notebooks/Job_postings

/content/drive/MyDrive/Colab_Notebooks/Job_postings


In [ ]:
!ls
!ls data/annotation
!ls model

data  model  models  model_training.ipynb
annoted_set.parquet
train_no_chunk.py  train.py


In [ ]:
!pip install -U transformers accelerate datasets scikit-learn iterative-stratification pyarrow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 144.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 48.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 144.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 55.0 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: scikit-learn
    Found existing installation: scikit-learn 1.6.1
    Uninstalling scikit-learn-1.6.1:
      Successfully uninstalled scikit-learn-1.6.1
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


In [ ]:
INPUT_PATH = "data/annotation/annoted_set.parquet"
OUTPUT_DIR = "models/moderncamembert_job_features"

In [1]:
import re
import pandas as pd
from transformers import AutoTokenizer

MODEL_NAME = "almanach/moderncamembert-base"
INPUT_PATH = "data/annotation/annoted_set.parquet"
TEXT_COL = "description"

def minimal_clean(text):
    if pd.isna(text):
        return ""
    text = str(text).strip()
    text = re.sub(r"\s+", " ", text)
    return text

df = pd.read_parquet(INPUT_PATH)
df["full_text"] = df[TEXT_COL].apply(minimal_clean)
df = df[df["full_text"].str.len() > 0].copy()

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def count_tokens(text):
    return len(tokenizer(text, add_special_tokens=True, truncation=False)["input_ids"])

df["n_tokens"] = df["full_text"].apply(count_tokens)

print(df["n_tokens"].describe(percentiles=[0.5, 0.75, 0.9, 0.95, 0.99]))

for limit in [256, 512, 768, 1024, 2048, 4096, 8192]:
    pct = (df["n_tokens"] <= limit).mean() * 100
    print(f"Offres <= {limit:>4} tokens : {pct:.1f}%")

FileNotFoundError: [Errno 2] No such file or directory: 'data/annotation/annoted_set.parquet'

In [ ]:
!python model/train_no_chunk.py

Nombre d'offres annotées utilisables : 7,300

Prévalence des labels :
career_progression      0.305205
non_salary_benefits     0.248082
company_culture         0.239452
work_meaning_impact     0.129315
schedule_flexibility    0.093836
dtype: float64

Calcul des longueurs en tokens sur les offres complètes...
[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (1306 > 1024). Running this sequence through the model will result in indexing errors

Statistiques longueur en tokens, offres complètes :
count    7300.000000
mean      370.039589
std       256.434556
min        20.000000
50%       320.000000
75%       487.000000
90%       680.000000
95%       816.000000
99%      1124.010000
max      6308.000000
Name: n_tokens, dtype: float64

Couverture selon MAX_LENGTH :
Offres <=  512 tokens : 77.7%
Offres <= 1024 tokens : 98.2%
Offres <= 2048 tokens : 99.9%
Offres <= 4096 tokens : 100.0%
Offres <= 8192 tokens : 100.0%

Offres qui se

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import torch

from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForSequenceClassification

In [ ]:
MODEL_DIR = "models/moderncamembert_job_features_full_text"
VALID_PATH = os.path.join(MODEL_DIR, "valid_split.parquet")
OUTPUT_ERRORS_PATH = os.path.join(MODEL_DIR, "valid_error_analysis.parquet")

In [ ]:
with open(os.path.join(MODEL_DIR, "labels.json"), "r", encoding="utf-8") as f:
    LABELS = json.load(f)

with open(os.path.join(MODEL_DIR, "thresholds.json"), "r", encoding="utf-8") as f:
    thresholds = json.load(f)

print("Labels :", LABELS)
print("Thresholds ", thresholds)

Labels : ['career_progression', 'company_culture', 'non_salary_benefits', 'work_meaning_impact', 'schedule_flexibility']
Thresholds : {'career_progression': 0.34, 'company_culture': 0.31000000000000005, 'non_salary_benefits': 0.43000000000000005, 'work_meaning_impact': 0.17000000000000004, 'schedule_flexibility': 0.29000000000000004}


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR)
model.to(device)
model.eval()

Device: cuda


Loading weights:   0%|          | 0/138 [00:00<?, ?it/s]

ModernBertForSequenceClassification(
  (model): ModernBertModel(
    (embeddings): ModernBertEmbeddings(
      (tok_embeddings): Embedding(32768, 768, padding_idx=0)
      (norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (drop): Dropout(p=0.0, inplace=False)
    )
    (layers): ModuleList(
      (0): ModernBertEncoderLayer(
        (attn_norm): Identity()
        (attn): ModernBertAttention(
          (Wqkv): Linear(in_features=768, out_features=2304, bias=False)
          (Wo): Linear(in_features=768, out_features=768, bias=False)
          (out_drop): Identity()
        )
        (mlp_norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): ModernBertMLP(
          (Wi): Linear(in_features=768, out_features=2304, bias=False)
          (act): GELUActivation()
          (drop): Dropout(p=0.0, inplace=False)
          (Wo): Linear(in_features=1152, out_features=768, bias=False)
        )
      )
      (1-21): 21 x ModernBertEncoderLayer(
        (at

In [ ]:
valid_df = pd.read_parquet(VALID_PATH)

print(valid_df.shape)
display(valid_df.head())

(1095, 8)


,offer_id,text,n_tokens,career_progression,company_culture,non_salary_benefits,work_meaning_impact,schedule_flexibility
0,10,"rattaché au responsable comptable groupe, vous...",185,0,0,0,0,0
1,11,vous aurez pour mission principale de particip...,192,1,0,0,0,0
2,18,description du poste : au sein de la direction...,420,0,0,0,0,0
3,21,"py33, cabinet d'expertise comptable à taille h...",324,0,0,0,0,0
4,40,le crédit agricole d'ile-de-france est la 1ère...,602,0,1,0,0,0


In [ ]:
def sigmoid(x):
    x = np.clip(x, -50, 50)
    return 1 / (1 + np.exp(-x))


def predict_proba_texts(texts, batch_size=16, max_length=1024):
    all_probs = []

    for start in tqdm(range(0, len(texts), batch_size)):
        batch_texts = texts[start:start + batch_size]

        enc = tokenizer(
            batch_texts,
            truncation=True,
            max_length=max_length,
            padding=True,
            return_tensors="pt",
        )

        enc = {k: v.to(device) for k, v in enc.items()}

        with torch.no_grad():
            outputs = model(**enc)
            logits = outputs.logits.detach().cpu().numpy()

        probs = sigmoid(logits)
        all_probs.append(probs)

    return np.vstack(all_probs)

In [ ]:
BATCH_SIZE = 16
MAX_LENGTH = 1024

valid_probs = predict_proba_texts(
    valid_df["text"].tolist(),
    batch_size=BATCH_SIZE,
    max_length=MAX_LENGTH,
)

valid_probs.shape

  0%|          | 0/69 [00:00<?, ?it/s]

(1095, 5)

In [ ]:
analysis_df = valid_df.copy()

for i, label in enumerate(LABELS):
    threshold = thresholds[label]

    analysis_df[f"proba_{label}"] = valid_probs[:, i]
    analysis_df[f"pred_{label}"] = (valid_probs[:, i] >= threshold).astype(int)
    analysis_df[f"true_{label}"] = analysis_df[label].astype(int)

    analysis_df[f"error_{label}"] = analysis_df[f"pred_{label}"] - analysis_df[f"true_{label}"]

    analysis_df[f"is_fp_{label}"] = (
        (analysis_df[f"pred_{label}"] == 1)
        & (analysis_df[f"true_{label}"] == 0)
    )

    analysis_df[f"is_fn_{label}"] = (
        (analysis_df[f"pred_{label}"] == 0)
        & (analysis_df[f"true_{label}"] == 1)
    )

    analysis_df[f"is_tp_{label}"] = (
        (analysis_df[f"pred_{label}"] == 1)
        & (analysis_df[f"true_{label}"] == 1)
    )

    analysis_df[f"is_tn_{label}"] = (
        (analysis_df[f"pred_{label}"] == 0)
        & (analysis_df[f"true_{label}"] == 0)
    )

    # Distance au seuil : utile pour trouver les cas ambigus.
    analysis_df[f"distance_to_threshold_{label}"] = (
        analysis_df[f"proba_{label}"] - threshold
    ).abs()

analysis_df.to_parquet(OUTPUT_ERRORS_PATH, index=False)

print("Saved:", OUTPUT_ERRORS_PATH)
display(analysis_df.head())

Saved: models/moderncamembert_job_features_full_text/valid_error_analysis.parquet


,offer_id,text,n_tokens,career_progression,company_culture,non_salary_benefits,work_meaning_impact,schedule_flexibility,proba_career_progression,pred_career_progression,...,distance_to_threshold_work_meaning_impact,proba_schedule_flexibility,pred_schedule_flexibility,true_schedule_flexibility,error_schedule_flexibility,is_fp_schedule_flexibility,is_fn_schedule_flexibility,is_tp_schedule_flexibility,is_tn_schedule_flexibility,distance_to_threshold_schedule_flexibility
0,10,"rattaché au responsable comptable groupe, vous...",185,0,0,0,0,0,0.058089,0,...,0.123500,0.074482,0,0,0,False,False,False,True,0.215518
1,11,vous aurez pour mission principale de particip...,192,1,0,0,0,0,0.891169,1,...,0.165854,0.046812,0,0,0,False,False,False,True,0.243188
2,18,description du poste : au sein de la direction...,420,0,0,0,0,0,0.029150,0,...,0.130826,0.015590,0,0,0,False,False,False,True,0.274410
3,21,"py33, cabinet d'expertise comptable à taille h...",324,0,0,0,0,0,0.066247,0,...,0.146119,0.018575,0,0,0,False,False,False,True,0.271425
4,40,le crédit agricole d'ile-de-france est la 1ère...,602,0,1,0,0,0,0.050275,0,...,0.016581,0.004331,0,0,0,False,False,False,True,0.285669


In [ ]:
data = analysis_df[analysis_df['is_fp_company_culture']==1]
data.iloc[2]

,25
offer_id,180
text,description du poste :a propos de lentreprise ...
n_tokens,261
career_progression,1
company_culture,0
non_salary_benefits,0
work_meaning_impact,0
schedule_flexibility,0
proba_career_progression,0.38263
pred_career_progression,1


In [ ]:
data.iloc[3]['text']

"description du poste :groupe industriel spécialisé de taille humaine, nous sommes présents en france et dans plus de 20 pays. nous concevons, fabriquons, installons et exploitons une large gamme d'appareils de chauffage gaz décentralisé et de production d'eau chaude sanitaire à usage industriel et tertiaire.afin de poursuivre notre belle histoire (nous fêtons nos 75 ans !), nous recherchons notre :ingénieur développement chauffage f/hpme / systèmes de chauffage industriel et tertiaire / poste polyvalent et évolutifnous mettons un point d'honneur à livrer des produits sûrs, fiables et techniquement à la pointe, et c'est grâce à vous que nous continuerons à le faire !conjuguant commerce et technique, votre rôle est central : vous êtes le moteur de l'évolution de nos gammes, dont vous garantissez la conformité, et vous venez en support des services clé de l'entreprise (production, vente, sav, approvisionnements).en veille technique constante, vous anticipez l'obsolescence des composants,